# Sentiment Analysis

Ensure that FastAPI is up and running. [instructions here](../fast_api/readme.md)

# Table of contents

1. Data prep
2. Standard
        2.1. Train Model
        2.2. Predict Labels
        2.3. Evaluate Model
3. NeXT
        3.1. Soft Match
            3.1.1. Train Model
            3.1.2. Predict Labels
            3.1.3. Evaluate Model
        3.2. Strict Match
            3.2.1. Train Model
            3.2.2. Predict Labels
            3.2.3. Evaluate Model

In [1]:
# imports
import requests
import json

In [2]:
FAST_API_URL = "http://localhost:9000"

 ---
 # 1. Data prep

We use a subset of [IMDB sentiment analysis](https://ai.stanford.edu/~amaas/data/sentiment/) dataset in this demo.

#### load smaple data from files

In [3]:
# load sample data from file
sa_data = json.load(open('data/sa_data.json'))
sa_data.keys()

dict_keys(['unlabeled_data', 'explanation_triples'])

---
#### Prepare data for Standard training

In [4]:
# load labelled data without explanations for Standard training
sa_standard_labeled_data = list(map(
    lambda x: {'text': x['text'], 'label': x['label']}, 
    sa_data['explanation_triples']
))
sa_standard_labeled_data[0]

{'text': 'Walt Disney\'s CINDERELLA takes a story everybody\'s familiar with and embellishes it with humor and suspense, while retaining the tale\'s essential charm. Disney\'s artists provide the film with an appealing storybook look that emanates delectable fairy tale atmosphere. It is beautifully, if conventionally, animated; the highlight being the captivating scene where the Fairy Godmother transforms a pumpkin into a majestic coach and Cinderella\'s rags to a gorgeous gown. Mack David, Al Hoffman, and Jerry Livingston provide lovely songs like A Dream Is a Wish Your Heart Makes" and "Bibbidi-Bobbidi-Boo" that enhance both the scenario and the characters.<br /><br />Even though CINDERELLA\'s story is predictable',
 'label': 'Positive'}

---
#### Prepare data for NeXT training

In [5]:
# load unlabelled data for NeXT training
sa_next_unlabeled_data = sa_data['unlabeled_data']
sa_next_unlabeled_data[0]

'The idea of making a film about the Beatles sounds doomed idea, as no production can catch the idea of the actual historic Beatles. Then it is perhaps best not to try to recreate the past, but to produce an illustration that works best with the other available Beatles material. This is exactly what \'Birth of the Beatles\' offers to us, the simple story known to us without any extravaganza.<br /><br />*** SPOILERS here on *** <br /><br />Be warned that not everything is that accurate as some Beatles-graduates might expect. The Beatles are seen performing songs that hardly were even composed by that time. The Beatles perform Ask Me Why"'

In [6]:
# load labelled data with explanations for NeXT training
sa_next_explanation_triples = sa_data['explanation_triples']
sa_next_explanation_triples[0]

{'text': 'Walt Disney\'s CINDERELLA takes a story everybody\'s familiar with and embellishes it with humor and suspense, while retaining the tale\'s essential charm. Disney\'s artists provide the film with an appealing storybook look that emanates delectable fairy tale atmosphere. It is beautifully, if conventionally, animated; the highlight being the captivating scene where the Fairy Godmother transforms a pumpkin into a majestic coach and Cinderella\'s rags to a gorgeous gown. Mack David, Al Hoffman, and Jerry Livingston provide lovely songs like A Dream Is a Wish Your Heart Makes" and "Bibbidi-Bobbidi-Boo" that enhance both the scenario and the characters.<br /><br />Even though CINDERELLA\'s story is predictable',
 'label': 'Positive',
 'explanation': "The word 'embellishes' appears in the text"}

---
#### Prepare data for evaluation

In [7]:
# load evaluation samples
sa_eval_data = json.load(open('data/sa_eval_data.json'))
# [Sentence, Label] pairs
sa_eval_data[0]

["This film is underrated. I loved it. It was truly sweet and heartfelt. A family who struggles but isn't made into a dysfunctional family which is so typical of films today. The film didn't make it an issue that they have little money or are Dominican Republican the way Hollywood have.<br /><br />Instead the issue is Victor is immature and needs to grow up. He does, slowly, by the film's end. He has a ways to go, but it was a heartfelt attempt to move forward. His grandmother is very cute and the scene where the little boy throws up had me laughing for the longest time. A truly heartfelt indie",
 'Positive']

---
#### Prepare data for prediction

In [8]:
sa_predict_data = list(map(lambda x: x[0], sa_eval_data))
sa_predict_data[0]

"This film is underrated. I loved it. It was truly sweet and heartfelt. A family who struggles but isn't made into a dysfunctional family which is so typical of films today. The film didn't make it an issue that they have little money or are Dominican Republican the way Hollywood have.<br /><br />Instead the issue is Victor is immature and needs to grow up. He does, slowly, by the film's end. He has a ways to go, but it was a heartfelt attempt to move forward. His grandmother is very cute and the scene where the little boy throws up had me laughing for the longest time. A truly heartfelt indie"

---
#### Define label space

In [9]:
# dictionary of (key: String, value: Int), key being the label and value being unique integer ID
sa_label_space = {
    "Positive": 0,
    "Negative": 1
}

---
---
# 2. Standard

Check [this](../fast_api/json_schema.py#L516) json schema for a list of all parameters.

Check [this](../model_training/internal_api/defaults.py) for default values.

## 2.1. Train model

### Define training parameters

In [10]:
sa_standard_train_params = {
    # a string name representing the model name
    "experiment_name": "imdb_sa_nle_standard",
    # a string name representing the dataset name
    "dataset_name": "imdb_sa_nle_standard",
    # task type - "sa" for Sentiment Analysis    
    "task": "sa",
    # training batch size
    "match_batch_size": 50,
    # learning rate
    "learning_rate": 0.1,
    # number of epochs
    "epochs": 5,
    # embedding to be used for training. usual default: "glove.840B.300d"
    "embeddings": "charngram.100d",
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 100,
    # random seed
    "random_state": 7698,
    # when "True" data has to be passed, 
    # "False" when re-training or the data was processed earlier and can be retrieved
    "build_data": True
}

### Run model training

In [11]:
# depending on input size, and computing environment this might take time.
# please check FAST API logs for updates
response = requests.post(
    FAST_API_URL + '/training/standard/api/',
    json={
        'params': sa_standard_train_params,
        'label_space': sa_label_space,
        'labeled_data': sa_standard_labeled_data,
    }
)
# JSON with "save_path" key is returned when successful
response.text

'{"save_path":"/home/Projects/ink-lab-lean-life/LEAN-LIFE/model_api/fast_api/../model_training/internal_api/../next_framework/data/saved_models/Clf_imdb_sa_nle_standard.p"}'

---
## 2.2. Predict labels

### Define prediction parameters

In [12]:
sa_standard_predict_params = {
    # a string name representing the model name
    "experiment_name": "imdb_sa_nle_standard",
    # a string name representing the dataset name
    "dataset_name": "imdb_sa_nle_standard",
    # task type - "sa" for Sentiment Analysis
    "task": "sa",
    # evaluation batch size
    "eval_batch_size": 50,
    # embedding to be used for training. usual default: "glove.840B.300d"
    "embeddings": "charngram.100d",
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 100
}

### Fetch predictions

In [13]:
response = requests.post(
    FAST_API_URL + '/training/standard/predict/',
    json={
        'params': sa_standard_predict_params,
        'label_space': sa_label_space,
        'prediction_data': sa_predict_data,
    }
)
# returns JSON object with keys
# "class_probs" - list of list representing predictions for each label
# "class_preds" - list of integers representing the label that was predicted
response.text

'{"class_probs":[[0.48354560136795044,0.5164543986320496],[0.48343217372894287,0.5165678858757019],[0.4839431643486023,0.5160568356513977],[0.48435676097869873,0.515643298625946],[0.47995486855506897,0.5200451612472534],[0.4862929880619049,0.5137070417404175],[0.4845595359802246,0.5154405236244202],[0.4825523793697357,0.5174476504325867],[0.47728413343429565,0.5227158069610596],[0.48190197348594666,0.5180981159210205]],"class_preds":[1,1,1,1,1,1,1,1,1,1]}'

---
## 2.3. Evaluate model

### Define evaluation parameters

In [14]:
sa_standard_eval_params = {
    # a string name representing the model name
    "experiment_name": "imdb_sa_nle_standard",
    # a string name representing the dataset name
    "dataset_name": "imdb_sa_nle_standard",
    # task type - "sa" for Sentiment Analysis
    "task": "sa",
    # evaluation batch size
    "eval_batch_size": 50,
    # embedding to be used for training. usual default: "glove.840B.300d"
    "embeddings": "charngram.100d",
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 100
}

### Run model evaluation

In [15]:
response = requests.post(
    FAST_API_URL + '/training/standard/eval/',
    json={
        'params': sa_standard_eval_params,
        'label_space': sa_label_space,
        'eval_data': sa_eval_data,
    }
)
# returns JSON object with keys
# "avg_loss" and "avg_eval_f1_score"
response.text

'{"avg_loss":0.68117,"avg_eval_f1_score":0.7}'

---
---
# 3. NeXT

Check [this](../fast_api/json_schema.py#L293) json schema for a list of all parameters.

Check [this](../model_training/internal_api/defaults.py) for default values.

## 3.1. Soft Match

## 3.1.1. Train model

### Define training parameters

In [16]:
sa_next_soft_match_params = {
    # a string name representing the model name
    "experiment_name": "imdb_sa_nle_soft_match",
    # a string name representing the dataset name
    "dataset_name": "imdb_sa_nle_soft_match",
    # integer representing the unlabelled dataset size
    "dataset_size": len(sa_next_unlabeled_data),
    # task type - "sa" for Sentiment Analysis    
    "task": "sa",
    # the total batch_size is "match_batch_size" + "unlabeled_batch_size"
    "match_batch_size": 50,
    "unlabeled_batch_size": 100,
    # learning rate
    "learning_rate": 0.1,
    # number of epochs
    "epochs": 5,
    # embedding to be used for training. usual default: "glove.840B.300d"
    "embeddings": "charngram.100d", # usual default: "glove.840B.300d"
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # soft_match loss weightage
    "gamma": 0.5,
    # number of hidden dimensions
    "hidden_dim": 100,
    # random seed
    "random_state": 7698,
    # FIND module number of hidden dimensions
    "pre_train_hidden_dim": 300,
    # FIND module training_size
    "pre_train_training_size": 50000,
    
    # enable soft match
    "soft_match": True,
    
    # "find" to train only FIND module, 
    # "clf" to train only the classifier,
    # "both" to train FIND module + classifier
    "stage": "both",
    
    # for the FIND module,
    # when "True" data has to be passed, 
    # "False" when re-training or the data was processed earlier and can be retrieved
    "pre_train_build_data": True,
    
    # when "True" data has to be passed, 
    # "False" when re-training or the data was processed earlier and can be retrieved
    "build_data": True
}

### Run model training

In [17]:
# depending on input size, and computing environment this might take time.
# please check FAST API logs for updates
response = requests.post(
    FAST_API_URL + '/training/next/api/',
    json={
        'params': sa_next_soft_match_params,
        'label_space': sa_label_space,
        'explanation_triples': sa_next_explanation_triples,
        'unlabeled_text': sa_next_unlabeled_data,
    }
)
# JSON with "save_path" key is returned when successful
response.text

'{"save_path":"/home/Projects/ink-lab-lean-life/LEAN-LIFE/model_api/fast_api/../model_training/internal_api/../next_framework/data/saved_models/Next-Clf_imdb_sa_nle_soft_match.p"}'

---
## 3.1.2. Predict labels

### Define prediction parameters

In [18]:
sa_next_soft_predict_params = {
    # a string name representing the model name
    "experiment_name": "imdb_sa_nle_soft_match",
    # a string name representing the dataset name
    "dataset_name": "imdb_sa_nle_soft_match",
    # integer representing the unlabelled dataset size
    "train_dataset_size": len(sa_next_unlabeled_data),
    # task type - "sa" for Sentiment Analysis
    "task": "sa",
    # evaluation batch size
    "eval_batch_size": 50,
    # embedding to be used for training. usual default: "glove.840B.300d"
    "embeddings": "charngram.100d", # usual default: "glove.840B.300d"
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 100,
    # FIND module training_size
    "pre_train_training_size": 50000
}

### Fetch predictions

In [19]:
response = requests.post(
    FAST_API_URL + '/training/next/predict/',
    json={
        'params': sa_next_soft_predict_params,
        'label_space': sa_label_space,
        'prediction_data': sa_predict_data,
    }
)
# returns JSON object with keys
# "class_probs" - list of list representing predictions for each label
# "class_preds" - list of integers representing the label that was predicted
response.text

'{"class_probs":[[0.3555419445037842,0.6444580554962158],[0.3592204749584198,0.640779435634613],[0.3575055003166199,0.6424945592880249],[0.36181148886680603,0.6381885409355164],[0.36266303062438965,0.6373370289802551],[0.3691120147705078,0.6308879256248474],[0.3589755892753601,0.6410244107246399],[0.36147555708885193,0.6385244131088257],[0.35292235016822815,0.6470776200294495],[0.3614174425601959,0.6385825276374817]],"class_preds":[1,1,1,1,1,1,1,1,1,1]}'

---
## 3.1.3. Evaluate model

### Define evaluation parameters

In [20]:
sa_next_soft_eval_params = {
    # a string name representing the model name
    "experiment_name": "imdb_sa_nle_soft_match",
    # a string name representing the dataset name
    "dataset_name": "imdb_sa_nle_soft_match",
    # integer representing the unlabelled dataset size
    "train_dataset_size": len(sa_next_unlabeled_data),
    # task type - "sa" for Sentiment Analysis
    "task": "sa",
    # evaluation batch size
    "eval_batch_size": 50,
    # embedding to be used for training. usual default: "glove.840B.300d"
    "embeddings": "charngram.100d", # usual default: "glove.840B.300d"
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 100,
    # FIND module training_size
    "pre_train_training_size": 50000
}

### Run model evaluation

In [21]:
response = requests.post(
    FAST_API_URL + '/training/next/eval/',
    json={
        'params': sa_next_soft_eval_params,
        'label_space': sa_label_space,
        'eval_data': sa_eval_data,
    }
)
# returns JSON object with keys
# "avg_loss", "avg_eval_ent_f1_score", "avg_eval_ent_f1_score" and "no_relation_thresholds"
response.text

'{"avg_loss":0.61921,"avg_eval_ent_f1_score":0.0,"avg_eval_val_f1_score":0.7,"no_relation_thresholds":[0.0,-1.0]}'

---
---
## 3.2. Strict Match

## 3.2.1. Train model

### Define training parameters

In [22]:
# parameters are same as in "soft_match" except "soft_match" is set to "False" here
sa_next_strict_match_params = {
    # a string name representing the model name
    "experiment_name": "imdb_sa_nle_strict_match",
    # a string name representing the dataset name
    "dataset_name": "imdb_sa_nle_strict_match",
    # integer representing the unlabelled dataset size
    "dataset_size": len(sa_next_unlabeled_data),
    # task type - "sa" for Sentiment Analysis    
    "task": "sa",
    # the total batch_size is "match_batch_size" + "unlabeled_batch_size"
    "match_batch_size": 50,
    "unlabeled_batch_size": 100,
    # learning rate
    "learning_rate": 0.1,
    # number of epochs
    "epochs": 5,
    # embedding to be used for training. usual default: "glove.840B.300d"
    "embeddings": "charngram.100d", # usual default: "glove.840B.300d"
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # soft_match loss weightage
    "gamma": 0.5,
    # number of hidden dimensions
    "hidden_dim": 100,
    # random seed
    "random_state": 7698,
    # FIND module number of hidden dimensions
    "pre_train_hidden_dim": 300,
    # FIND module training_size
    "pre_train_training_size": 50000,
    
    # enable soft match
    "soft_match": False,
    
    # "find" to train only FIND module, 
    # "clf" to train only the classifier,
    # "both" to train FIND module + classifier
    "stage": "both",
    
    # for the FIND module,
    # when "True" data has to be passed, 
    # "False" when re-training or the data was processed earlier and can be retrieved
    "pre_train_build_data": True,
    
    # when "True" data has to be passed, 
    # "False" when re-training or the data was processed earlier and can be retrieved
    "build_data": True
}

### Run model training

In [23]:
# depending on input size, and computing environment this might take time.
# please check FAST API logs for updates
response = requests.post(
    FAST_API_URL + '/training/next/api/',
    json={
        'params': sa_next_strict_match_params,
        'label_space': sa_label_space,
        'explanation_triples': sa_next_explanation_triples,
        'unlabeled_text': sa_next_unlabeled_data,
    }
)
# JSON with "save_path" key is returned when successful
response.text

'{"save_path":"/home/Projects/ink-lab-lean-life/LEAN-LIFE/model_api/fast_api/../model_training/internal_api/../next_framework/data/saved_models/Clf_imdb_sa_nle_strict_match.p"}'

---
## 3.2.2. Predict labels

### Define prediction parameters

In [24]:
sa_next_strict_predict_params = {
    # a string name representing the model name
    "experiment_name": "imdb_sa_nle_strict_match",
    # a string name representing the dataset name
    "dataset_name": "imdb_sa_nle_strict_match",
    # task type - "sa" for Sentiment Analysis
    "task": "sa",
    # evaluation batch size
    "eval_batch_size": 50,
    # embedding to be used for training. usual default: "glove.840B.300d"
    "embeddings": "charngram.100d",
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 100
}

### Fetch predictions

In [25]:
response = requests.post(
    FAST_API_URL + '/training/standard/predict/',
    json={
        'params': sa_next_strict_predict_params,
        'label_space': sa_label_space,
        'prediction_data': sa_predict_data,
    }
)
# returns JSON object with keys
# "class_probs" - list of list representing predictions for each label
# "class_preds" - list of integers representing the label that was predicted
response.text

'{"class_probs":[[0.04335974529385567,0.9566402435302734],[0.0482737272977829,0.9517263174057007],[0.04450971260666847,0.955490231513977],[0.04930686950683594,0.9506931304931641],[0.05088886246085167,0.9491111636161804],[0.05178869515657425,0.9482112526893616],[0.04710967466235161,0.9528903365135193],[0.04865354672074318,0.9513464570045471],[0.04324809089303017,0.9567518830299377],[0.05046840012073517,0.9495315551757812]],"class_preds":[1,1,1,1,1,1,1,1,1,1]}'

---
## 3.2.3. Evaluate model

### Define evaluation parameters

In [26]:
sa_next_strict_eval_params = {
    # a string name representing the model name
    "experiment_name": "imdb_sa_nle_strict_match",
    # a string name representing the dataset name
    "dataset_name": "imdb_sa_nle_strict_match",
    # task type - "sa" for Sentiment Analysis
    "task": "sa",
    # evaluation batch size
    "eval_batch_size": 50,
    # embedding to be used for training. usual default: "glove.840B.300d"
    "embeddings": "charngram.100d",
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 100
}

### Run model evaluation

In [27]:
response = requests.post(
    FAST_API_URL + '/training/standard/eval/',
    json={
        'params': sa_next_strict_eval_params,
        'label_space': sa_label_space,
        'eval_data': sa_eval_data,
    }
)
# returns JSON object with keys
# "avg_loss" and "avg_eval_f1_score"
response.text

'{"avg_loss":0.94424,"avg_eval_f1_score":0.7}'